# Gold Salary Trends Mart

**Purpose**: Dashboard-ready salary trends with percentile distributions by sector, role, and location.

**Target Table**: `workspace.gold.gold_salary_trends`

**Grain**: One row per date per sector/role/location combination with salary statistics

**Key Metrics**:
- Median salaries (min and max)
- Percentile distributions (P25, P75, P90)
- Observation counts
- Period-over-period salary changes

**Data Sources**:
- `workspace.warehouse.fact_salary`

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_salary_trends
USING DELTA
COMMENT 'Salary trends with percentile distributions for dashboard consumption'
AS

WITH salary_base AS (
  SELECT
    fs.observation_date_sk AS salary_date_sk,
    fs.sector_sk,
    fs.role_sk,
    fs.location_sk,
    fs.company_sk,
    fs.salary_min,
    fs.salary_max,
    fs.salary_currency,
    fs.salary_period,
    fs.salary_confidence
  FROM workspace.warehouse.fact_salary fs
  WHERE fs.observation_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 365), 'yyyyMMdd') AS INT)
    AND fs.salary_period = 'ANNUAL'
    AND fs.salary_currency = 'USD'
    AND fs.salary_min IS NOT NULL
    AND fs.salary_max IS NOT NULL
    AND fs.salary_min > 0
    AND fs.salary_max > 0
),

base_metrics AS (
  SELECT
    salary_date_sk,
    sector_sk,
    role_sk,
    location_sk,
    company_sk,
    
    -- MEASURES: Salary statistics
    PERCENTILE(salary_min, 0.50) AS salary_min_median,
    PERCENTILE(salary_max, 0.50) AS salary_max_median,
    PERCENTILE(salary_min, 0.25) AS salary_min_p25,
    PERCENTILE(salary_min, 0.75) AS salary_min_p75,
    PERCENTILE(salary_max, 0.25) AS salary_max_p25,
    PERCENTILE(salary_max, 0.75) AS salary_max_p75,
    PERCENTILE(salary_max, 0.90) AS salary_max_p90,
    
    MIN(salary_min) AS salary_min_floor,
    MAX(salary_max) AS salary_max_ceiling,
    
    ROUND(AVG(salary_min), 2) AS salary_min_avg,
    ROUND(AVG(salary_max), 2) AS salary_max_avg,
    ROUND(AVG(salary_confidence), 4) AS avg_confidence,
    
    COUNT(*) AS observation_count
    
  FROM salary_base
  GROUP BY salary_date_sk, sector_sk, role_sk, location_sk, company_sk
),

-- Rollup: By Sector (all roles, all locations, all companies)
sector_agg AS (
  SELECT
    salary_date_sk,
    sector_sk,
    CAST(NULL AS BIGINT) AS role_sk,
    CAST(NULL AS BIGINT) AS location_sk,
    CAST(NULL AS BIGINT) AS company_sk,
    
    PERCENTILE(salary_min_median, 0.50) AS salary_min_median,
    PERCENTILE(salary_max_median, 0.50) AS salary_max_median,
    PERCENTILE(salary_min_p25, 0.50) AS salary_min_p25,
    PERCENTILE(salary_min_p75, 0.50) AS salary_min_p75,
    PERCENTILE(salary_max_p25, 0.50) AS salary_max_p25,
    PERCENTILE(salary_max_p75, 0.50) AS salary_max_p75,
    PERCENTILE(salary_max_p90, 0.50) AS salary_max_p90,
    MIN(salary_min_floor) AS salary_min_floor,
    MAX(salary_max_ceiling) AS salary_max_ceiling,
    ROUND(AVG(salary_min_avg), 2) AS salary_min_avg,
    ROUND(AVG(salary_max_avg), 2) AS salary_max_avg,
    ROUND(AVG(avg_confidence), 4) AS avg_confidence,
    SUM(observation_count) AS observation_count
  FROM base_metrics
  GROUP BY salary_date_sk, sector_sk
),

-- Rollup: By Role (all sectors, all locations)
role_agg AS (
  SELECT
    salary_date_sk,
    CAST(-1 AS BIGINT) AS sector_sk,
    role_sk,
    CAST(NULL AS BIGINT) AS location_sk,
    CAST(NULL AS BIGINT) AS company_sk,
    
    PERCENTILE(salary_min_median, 0.50) AS salary_min_median,
    PERCENTILE(salary_max_median, 0.50) AS salary_max_median,
    PERCENTILE(salary_min_p25, 0.50) AS salary_min_p25,
    PERCENTILE(salary_min_p75, 0.50) AS salary_min_p75,
    PERCENTILE(salary_max_p25, 0.50) AS salary_max_p25,
    PERCENTILE(salary_max_p75, 0.50) AS salary_max_p75,
    PERCENTILE(salary_max_p90, 0.50) AS salary_max_p90,
    MIN(salary_min_floor) AS salary_min_floor,
    MAX(salary_max_ceiling) AS salary_max_ceiling,
    ROUND(AVG(salary_min_avg), 2) AS salary_min_avg,
    ROUND(AVG(salary_max_avg), 2) AS salary_max_avg,
    ROUND(AVG(avg_confidence), 4) AS avg_confidence,
    SUM(observation_count) AS observation_count
  FROM base_metrics
  GROUP BY salary_date_sk, role_sk
),

-- Rollup: By Location (all sectors, all roles)
location_agg AS (
  SELECT
    salary_date_sk,
    CAST(-1 AS BIGINT) AS sector_sk,
    CAST(NULL AS BIGINT) AS role_sk,
    location_sk,
    CAST(NULL AS BIGINT) AS company_sk,
    
    PERCENTILE(salary_min_median, 0.50) AS salary_min_median,
    PERCENTILE(salary_max_median, 0.50) AS salary_max_median,
    PERCENTILE(salary_min_p25, 0.50) AS salary_min_p25,
    PERCENTILE(salary_min_p75, 0.50) AS salary_min_p75,
    PERCENTILE(salary_max_p25, 0.50) AS salary_max_p25,
    PERCENTILE(salary_max_p75, 0.50) AS salary_max_p75,
    PERCENTILE(salary_max_p90, 0.50) AS salary_max_p90,
    MIN(salary_min_floor) AS salary_min_floor,
    MAX(salary_max_ceiling) AS salary_max_ceiling,
    ROUND(AVG(salary_min_avg), 2) AS salary_min_avg,
    ROUND(AVG(salary_max_avg), 2) AS salary_max_avg,
    ROUND(AVG(avg_confidence), 4) AS avg_confidence,
    SUM(observation_count) AS observation_count
  FROM base_metrics
  GROUP BY salary_date_sk, location_sk
),

-- Combine all aggregation levels
combined_agg AS (
  SELECT * FROM sector_agg
  UNION ALL
  SELECT * FROM role_agg
  UNION ALL
  SELECT * FROM location_agg
),

-- Add temporal metrics
temporal_enriched AS (
  SELECT
    ca.*,
    
    -- Prior period median salaries
    LAG(ca.salary_max_median, 30) OVER (
      PARTITION BY ca.sector_sk, ca.role_sk, ca.location_sk, ca.company_sk
      ORDER BY ca.salary_date_sk
    ) AS prev_month_salary_max_median,
    
    LAG(ca.salary_max_median, 90) OVER (
      PARTITION BY ca.sector_sk, ca.role_sk, ca.location_sk, ca.company_sk
      ORDER BY ca.salary_date_sk
    ) AS prev_quarter_salary_max_median
    
  FROM combined_agg ca
)

SELECT
  -- DIMENSIONS
  te.salary_date_sk,
  te.sector_sk,
  te.role_sk,
  te.location_sk,
  te.company_sk,
  
  -- MEASURES: Salary statistics
  CAST(te.salary_min_median AS DECIMAL(18,2)) AS salary_min_median,
  CAST(te.salary_max_median AS DECIMAL(18,2)) AS salary_max_median,
  CAST(te.salary_min_p25 AS DECIMAL(18,2)) AS salary_min_p25,
  CAST(te.salary_min_p75 AS DECIMAL(18,2)) AS salary_min_p75,
  CAST(te.salary_max_p25 AS DECIMAL(18,2)) AS salary_max_p25,
  CAST(te.salary_max_p75 AS DECIMAL(18,2)) AS salary_max_p75,
  CAST(te.salary_max_p90 AS DECIMAL(18,2)) AS salary_max_p90,
  CAST(te.salary_min_floor AS DECIMAL(18,2)) AS salary_min_floor,
  CAST(te.salary_max_ceiling AS DECIMAL(18,2)) AS salary_max_ceiling,
  CAST(te.salary_min_avg AS DECIMAL(18,2)) AS salary_min_avg,
  CAST(te.salary_max_avg AS DECIMAL(18,2)) AS salary_max_avg,
  te.avg_confidence,
  te.observation_count,
  
  -- DERIVED: Salary range
  CAST((te.salary_max_median - te.salary_min_median) AS DECIMAL(18,2)) AS salary_range_median,
  
  -- DERIVED: Change vs previous periods
  CASE 
    WHEN te.prev_month_salary_max_median > 0 
    THEN CAST((te.salary_max_median - te.prev_month_salary_max_median) AS DECIMAL(18,2))
    ELSE NULL 
  END AS delta_vs_prev_month,
  
  CASE 
    WHEN te.prev_month_salary_max_median > 0 
    THEN CAST((te.salary_max_median - te.prev_month_salary_max_median) AS DECIMAL(10,4)) / te.prev_month_salary_max_median
    ELSE NULL 
  END AS pct_change_vs_prev_month,
  
  -- METADATA
  CURRENT_TIMESTAMP() AS created_at,
  UUID() AS batch_id
  
FROM temporal_enriched te
WHERE te.observation_count >= 5  -- Minimum sample size for reliable statistics
ORDER BY te.salary_date_sk DESC;

In [0]:
%sql
-- Validation: Salary ranges by sector
SELECT
  s.sector_name,
  COUNT(DISTINCT gst.salary_date_sk) AS days_with_data,
  ROUND(AVG(gst.salary_min_median), 0) AS avg_salary_min_median,
  ROUND(AVG(gst.salary_max_median), 0) AS avg_salary_max_median,
  ROUND(AVG(gst.salary_max_p90), 0) AS avg_salary_p90,
  SUM(gst.observation_count) AS total_observations
FROM workspace.gold.gold_salary_trends gst
JOIN workspace.warehouse.dim_sector s ON gst.sector_sk = s.sector_sk
WHERE gst.role_sk IS NULL
  AND gst.location_sk IS NULL
  AND gst.company_sk IS NULL
GROUP BY s.sector_name
ORDER BY avg_salary_max_median DESC;